# Model Serving with TuiML

This tutorial covers TuiML's built-in model serving capabilities:
- **REST API Server** - Serve predictions via HTTP endpoints
- **Direct Model Serving** - Call `.serve()` on any trained model
- **CLI Integration** - `tuiml serve` command
- **Model Management** - Load, unload, and monitor models

TuiML makes it easy to deploy trained models as production-ready APIs.

## 2. Quick Start: Direct Model Serving

The simplest way to serve a model is calling `.serve()` directly on a trained model.

In [ ]:
# First, let's train a model
from tuiml.algorithms.bayesian import NaiveBayesClassifier
from tuiml.datasets import load_dataset
import numpy as np

# Load iris dataset
dataset = load_dataset("iris")
X, y = dataset.X, dataset.y

# Train a classifier
nb = NaiveBayesClassifier()
nb.fit(X, y)

print(f"Model trained: {nb.__class__.__name__}")
print(f"Training samples: {len(X)}")
print(f"Features: {X.shape[1]}")

### Serve the Model

**Note:** Running `.serve()` starts a blocking server. In a notebook, you'd typically run this in a separate cell or script.

```python
# This will start a REST API server
nb.serve(port=8000)

# Server running at http://127.0.0.1:8000
# API docs at http://127.0.0.1:8000/docs
```

In [ ]:
# Show the serve method signature
help(nb.serve)

## 3. High-Level API: `tuiml.serve()`

For serving saved models from disk, use the high-level API.

In [ ]:
import tuiml

# First, save the trained model
model_path = "demo_classifier.pkl"
tuiml.save(nb, model_path, metadata={"dataset": "iris", "accuracy": 0.96})

print(f"Model saved to: {model_path}")

### Serve from File

```python
# Serve the saved model
tuiml.serve("demo_classifier.pkl", port=8000)
```

## 4. CLI: `tuiml serve`

The CLI provides a convenient way to serve models from the command line.

```bash
# Basic usage
tuiml serve model.pkl

# With options
tuiml serve classifier.pkl --port 9000 --host 0.0.0.0 --model-id my_classifier

# Production with multiple workers
tuiml serve model.pkl --workers 4 --host 0.0.0.0
```

In [ ]:
# Show CLI help
!tuiml serve --help

## 5. REST API Endpoints

When the server is running, these endpoints are available:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Health check |
| `/stats` | GET | Server statistics |
| `/models` | GET | List loaded models |
| `/models` | POST | Load a new model |
| `/models/{id}` | GET | Get model info |
| `/models/{id}` | DELETE | Unload a model |
| `/models/{id}/predict` | POST | Make predictions |
| `/models/{id}/predict_proba` | POST | Get probabilities |
| `/predict` | POST | Predict with default model |

## 6. Making Predictions via API

Once the server is running, you can make predictions using HTTP requests.

### Using curl

```bash
# Make a prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [[5.1, 3.5, 1.4, 0.2], [6.2, 2.9, 4.3, 1.3]]}'

# Response:
# {
#   "predictions": [0, 1],
#   "model_id": "NaiveBayesClassifier",
#   "model_class": "NaiveBayesClassifier"
# }
```

### Using Python requests

```python
import requests

# Make prediction request
response = requests.post(
    "http://localhost:8000/predict",
    json={"features": [[5.1, 3.5, 1.4, 0.2]]}
)

result = response.json()
print(f"Prediction: {result['predictions']}")
```

### Get Probability Predictions

```bash
curl -X POST http://localhost:8000/models/NaiveBayesClassifier/predict_proba \
  -H "Content-Type: application/json" \
  -d '{"features": [[5.1, 3.5, 1.4, 0.2]]}'

# Response:
# {
#   "probabilities": [[0.95, 0.03, 0.02]],
#   "classes": [0, 1, 2],
#   "model_id": "NaiveBayesClassifier"
# }
```

## 7. ModelServer Class

For more control, use the `ModelServer` class directly.

In [ ]:
from tuiml.serving import ModelServer, ModelManager

# Create a server with custom settings
server = ModelServer(
    max_models=10,  # Maximum models to keep in memory
    title="My ML API",
    version="1.0.0"
)

print(f"Server title: {server.title}")
print(f"Max models: {server.manager.max_models}")

In [ ]:
# Load multiple models
server.load_model("naive_bayes", "demo_classifier.pkl")

# List loaded models
print(f"Loaded models: {server.manager.list_models()}")

In [ ]:
# Get model info
info = server.manager.get_model_info("naive_bayes")
print("Model Info:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
# Make predictions programmatically
test_data = [[5.1, 3.5, 1.4, 0.2], [6.2, 2.9, 4.3, 1.3]]
predictions = server.manager.predict("naive_bayes", test_data)

print(f"Predictions: {predictions}")

## 8. ModelManager for Advanced Use Cases

The `ModelManager` handles in-memory model caching with LRU eviction.

In [ ]:
from tuiml.serving import ModelManager

# Create a manager with limited cache
manager = ModelManager(max_models=5)

# Load a model
manager.load("classifier", "demo_classifier.pkl", metadata={"version": "1.0"})

print(f"Loaded: {manager.list_models()}")
print(f"Is loaded: {manager.is_loaded('classifier')}")

In [ ]:
# Get statistics
stats = manager.get_stats()
print("Manager Stats:")
print(f"  Loaded models: {stats['loaded_models']}")
print(f"  Max capacity: {stats['max_models']}")
print(f"  Total predictions: {stats['total_predictions']}")

In [ ]:
# Make predictions and check updated stats
for _ in range(10):
    manager.predict("classifier", [[5.1, 3.5, 1.4, 0.2]])

stats = manager.get_stats()
print(f"Total predictions after 10 calls: {stats['total_predictions']}")

In [ ]:
# Unload model
manager.unload("classifier")
print(f"Models after unload: {manager.list_models()}")

## 9. Complete Example: Train and Serve

A complete workflow from training to serving.

In [ ]:
import tuiml
from tuiml.algorithms.trees import C45TreeClassifier
from tuiml.datasets import load_dataset

# 1. Load data
dataset = load_dataset("iris")
X, y = dataset.X, dataset.y

# 2. Train model
model = C45TreeClassifier(confidence_factor=0.25)
model.fit(X, y)

# 3. Evaluate
accuracy = (model.predict(X) == y).mean()
print(f"Training accuracy: {accuracy:.2%}")

# 4. Save model
tuiml.save(model, "c45_classifier.pkl", metadata={"accuracy": accuracy})
print("Model saved!")

# 5. Serve (uncomment to run)
# model.serve(port=8000)
# Or: tuiml.serve("c45_classifier.pkl", port=8000)

## 10. API Documentation

When the server is running, automatic API documentation is available:

- **Swagger UI**: `http://localhost:8000/docs`
- **ReDoc**: `http://localhost:8000/redoc`

These provide interactive documentation where you can test endpoints directly.

## 11. Production Deployment Tips

### Multiple Workers
```bash
tuiml serve model.pkl --workers 4 --host 0.0.0.0
```

### Behind a Reverse Proxy (nginx)
```nginx
upstream tuiml {
    server 127.0.0.1:8000;
}

server {
    listen 80;
    location / {
        proxy_pass http://tuiml;
    }
}
```

### Docker Deployment
```dockerfile
FROM python:3.10-slim

WORKDIR /app
RUN pip install --no-cache-dir tuiml
COPY model.pkl .

EXPOSE 8000
CMD ["tuiml", "serve", "model.pkl", "--host", "0.0.0.0"]
```

## 12. Summary

TuiML provides multiple ways to serve trained models:

| Method | Use Case |
|--------|----------|
| `model.serve()` | Quick serving of in-memory models |
| `tuiml.serve()` | Serve saved model files |
| `tuiml serve` CLI | Command-line deployment |
| `ModelServer` class | Custom server configuration |
| `ModelManager` | Programmatic model management |

All methods provide:
- REST API with automatic OpenAPI documentation
- Support for multiple models
- Prediction and probability endpoints
- Health checks and statistics

In [ ]:
# Cleanup demo files
import os
for f in ["demo_classifier.pkl", "c45_classifier.pkl"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cleaned up: {f}")